This week retains the two-dimensional setting and changes the covariance assumption. We will fit the model, select how much to pool covariances, freeze it, fit one calibration parameter on separate data, and finally evaluate.
Use this exact generator, with $X$ of shape $(n,2)$ and integer $y$ of shape $(n,)$:
| Split | Size | Seed | Use |
|:-----|-----:|-----:|:----|
| Training | 240 | 20261109 | Fit means, proportions, covariances |
| Validation | 160 | 20261110 | Choose covariance pooling |
| Calibration | 160 | 20261111 | Fit one scalar after freezing the model |
| Test | 2000 | 20261112 | Final evaluation only |

In [ ]:
import numpy as np

def make_data(n, seed):
    rng = np.random.default_rng(seed)
    y = rng.binomial(1, 0.4, n)
    means = np.array([[-1., 0.], [1., 0.5]])
    covs = np.array([[[1., 0.5], [0.5, 1.]],
                     [[2., -0.6], [-0.6, 0.5]]])
    z = rng.standard_normal((n, 2))
    X = np.empty_like(z)
    for k in (0, 1):
        mask = (y == k)
        X[mask] = z[mask] @ np.linalg.cholesky(covs[k]).T + means[k]
    return X, y

No.6

Implement `fit_gaussian(X,y,rho,eps)` returning $(\hat\pi,\hat\mu,\widetilde\Sigma)$ with shapes scalar, $(2,d)$, and $(2,d,d)$. 

Implement `gaussian_score(model,Xnew)` returning shape $(m,).$

In [ ]:
def fit_gaussian(X, y, rho, eps):
    n, d = X.shape
    pi, mu = 0, np.zeros((2, d)) # The shape of μ and x is (1, d).
    for i in range(n):
        pi = pi + y[i]
        mu[y[i]] = mu[y[i]] + X[i]
    pi = pi/n
    mu[1] = mu[1]/(n*pi)
    mu[0] = mu[0]/(n*(1-pi))
    Sigma = np.zeros((2, d, d))
    for i in range(n):
        diff = X[i] - mu[y[i]]
        Sigma[y[i]] = Sigma[y[i]] + np.outer(diff, diff) # (μ - x)(μ - x)^\top
    Sigma[0] = Sigma[0]/(n * (1-pi))
    Sigma[1] = Sigma[1]/(n * pi)
    Sigmapool = (1-pi) * Sigma[0] + pi * Sigma[1]
    Sigma_tilde = np.zeros((2, d, d))
    Sigma_tilde[0] = (1-rho) * Sigma[0] + rho * Sigmapool + eps * np.identity(d)
    Sigma_tilde[1] = (1-rho) * Sigma[1] + rho * Sigmapool + eps * np.identity(d)
    return pi, mu, Sigma_tilde

def gaussian_score(model, Xnew):
    pi, mu, S = model
    m, d = Xnew.shape
    s = np.zeros(m)
    for i in range(m):
        L0, L1 = np.linalg.cholesky(S[0]), np.linalg.cholesky(S[1])
        logdet0, logdet1 = 2 * np.sum(np.log(np.diag(L0))), 2 * np.sum(np.log(np.diag(L1)))
        diff1, diff0 = np.linalg.solve(L1, Xnew[i] - mu[1]), np.linalg.solve(L0, Xnew[i] - mu[0])
        s[i] = -0.5 * (diff1.T @ diff1 - diff0.T @ diff0) + np.log(pi) - np.log(1-pi) - 0.5 * (logdet1 - logdet0)
    return s

#-------------- Test data ------------------

X, y = make_data(240, seed=20261109)

rho = 0.35
eps = 1e-3

model = fit_gaussian(X, y, rho, eps)

# 测试点：类均值附近、两类之间、远离均值
Xnew = np.array([
    [-1.0, 0.0],
    [ 1.0, 0.5],
    [ 0.0, 0.0],
    [ 2.0, 0.0],
    [-3.0, 0.0],
    [ 3.0, 1.0]
])

scores = gaussian_score(model, Xnew)

print("pi =", model[0])
print("mu =", model[1])
print("scores =", scores)
print("predictions =", (scores >= 0).astype(int))

assert np.shape(model[0]) == ()
assert model[1].shape == (2, 2)
assert model[2].shape == (2, 2, 2)
assert scores.shape == (6,)
assert np.all(np.isfinite(scores))

pi = 0.5

mu = np.zeros((2, 2))

S = np.array([
    [[1., 0.], [0., 1.]],
    [[4., 0.], [0., 4.]]
])

known_model = (pi, mu, S)

Xtest = np.array([
    [0., 0.],
    [2., 0.]
])

expected = np.array([
    -np.log(4),
    1.5 - np.log(4)
])

actual = gaussian_score(known_model, Xtest)

print("Expected:", expected)
print("Actual:  ", actual)
print("All tests passed!")

Fit candidates $\rho \in \{0, 0.5, 1\}$ on training data and choose the smallest validation mean log loss; break exact ties toward larger $\rho$. Retain both the selected model and the $\rho = 1$ baseline. Record validation error, log loss, and Brier loss. Do not refit on validation data.

In [ ]:
Xtrain, ytrain = make_data(240, seed=20261109)
model0 = fit_gaussian(Xtrain, ytrain, 0, eps)
model05 = fit_gaussian(Xtrain, ytrain, 0.5, eps)
model1 = fit_gaussian(Xtrain, ytrain, 1, eps)

def logloss(X, y, model): 
    n, d = X.shape
    l = 0
    s = gaussian_score(model, X)
    for i in range(n):
        l = l + np.logaddexp(0, s[i]) - y[i] * s[i]
    return l/n

def Brierloss(X, y, model):
    n, d = X.shape
    s = gaussian_score(model, X)
    l = 0
    for i in range(n):
        if -s[i] <= 0:
            p1 = np.exp(-np.logaddexp(0, -s[i]))
            l = l + (p1 - y[i])**2
        else:
            p0 = np.exp(-np.logaddexp(0, s[i]))
            l = l + (1 - p0 - y[i])**2
    return l/n
    
def Err(X, y, model):
    n, d = X.shape
    s = gaussian_score(model, X)
    e = 0
    for i in range(n):
        haty = 0
        if np.logaddexp(0, -s[i]) <= np.log(2):
            haty = 1
        if y[i] != haty:
            e = e + 1
    return e/n
               
XV, yV = make_data(160, seed=20261110)
print(logloss(XV, yV, model0), logloss(XV, yV, model05), logloss(XV, yV, model1), '\n')
# Using the validation samples, we claim that ρ = 0.5 is the optimal choice.

print(Brierloss(XV, yV, model05), Brierloss(XV, yV, model1))
print(Err(XV, yV, model05), Err(XV, yV, model1))

No. 7

Implement `scale_loss_grad(alpha, s, y)` returning loss, first derivative, and second derivative.

Check centered differences at $\alpha = 0.7$ on $s = (-2, -1, 1, 2)$, $y = (0, 1, 0, 1)$, with step $10^{-6}$ and absolute gradient error below $10^{-8}$. Check finite outputs for scores $\pm 1000$. Design an endpoint-aware bisection routine `fit_scale(s, y)` using 60 bisections; return 1 when all scores are zero. Give its arithmetic cost in $m$.

In [ ]:
def scale_loss_grad(alpha, s, y):
    m = s.shape[0]
    l, grad, H = 0, 0, 0
    for i in range(m):
        p = np.exp(-np.logaddexp(0, -alpha * s[i]))
        l = l + np.logaddexp(0, (1-2*y[i]) * alpha * s[i])
        grad = grad + (p - y[i]) * s[i]
        H = H + p * (1-p) * s[i]**2
    return l/m, grad/m, H/m

def fit_scale(s,y):
    if np.all(s == 0):
        return 1.0
    inf = scale_loss_grad(0.25, s, y)[1]
    sup = scale_loss_grad(4, s, y)[1]
    if inf >= 0:
        return 0.25
    if sup <= 0:
        return 4
    l, u = 0.25, 4
    for i in range(60):
        mid = (l+u)/2
        if scale_loss_grad(mid, s, y)[1] < 0:
            l = mid
        else:
            u = mid
    return (l+u)/2
    
#-------------- Test data ------------------

s = np.array([-2., -1., 1., 2.])
y = np.array([0, 1, 0, 1])

alpha = 0.7
h = 1e-6

loss, grad, H = scale_loss_grad(alpha, s, y)

loss_plus = scale_loss_grad(alpha+h, s, y)[0]
loss_minus = scale_loss_grad(alpha-h, s, y)[0]

grad_fd = (loss_plus - loss_minus) / (2*h)

error = abs(grad - grad_fd)

print("Analytical gradient:", grad)
print("Numerical gradient:", grad_fd)
print("Absolute error:", error)

print(error < 1e-8)


s_extreme = np.array([-1000., 1000.])
y_extreme = np.array([0, 1])

result = scale_loss_grad(0.7, s_extreme, y_extreme)

print(result)
assert np.all(np.isfinite(result))

Generate the calibration split and fit only $\alpha$. Explain why the calibration loss cannot exceed its value at $\alpha = 1$, up to numerical tolerance, but the same guarantee does not hold on test data. Freeze this scalar before generating the test set.

In [ ]:
Xcali, ycali = make_data(160, seed=20261111)

def temperScaling(model):
    s = gaussian_score(model, Xcali)
    return fit_scale(s, ycali)

hatalpha = temperScaling(model05)

s = gaussian_score(model05, Xcali)
Ca = scale_loss_grad(hatalpha, s, ycali)[0]
C1 = scale_loss_grad(1, s, ycali)[0]
print(Ca < C1)

Xtest, ytest = make_data(2000, seed=20261112)

s = gaussian_score(model05, Xtest)
Ca = scale_loss_grad(hatalpha, s, ytest)[0]
C1 = scale_loss_grad(1, s, ytest)[0]
print(Ca, C1, Ca < C1)

Report test error, log loss, and Brier loss for the pooled baseline, selected model, and scaled selected model. Compare baseline versus selected **classification errors** using Problem 5. For selected probabilities before and after scaling, also tabulate count, mean prediction, and observed positive fraction in the five fixed bins $[0,.2),\ldots,[.8,1]$; label empty bins as unavailable. Explain what finite bin counts, a slightly worse test log loss, and identical hard predictions do and do not imply.

In [ ]:
# pooled baseline: model1; selected model: model05; scaled selected model: Ca.
def BrierlossA(X, y, s, alpha):
    n, d = X.shape
    l = 0
    for i in range(n):
        if -s[i] <= 0:
            p1 = np.exp(-np.logaddexp(0, -alpha * s[i]))
            l = l + (p1 - y[i])**2
        else:
            p0 = np.exp(-np.logaddexp(0, alpha * s[i]))
            l = l + (1 - p0 - y[i])**2
    return l/n
    
def ErrA(X, y, s, alpha):
    n, d = X.shape
    e = 0
    for i in range(n):
        haty = 0
        if np.logaddexp(0, -alpha * s[i]) <= np.log(2):
            haty = 1
        if y[i] != haty:
            e = e + 1
    return e/n
               
print(logloss(Xtest, ytest, model05), Ca, logloss(Xtest, ytest, model1))
print(Brierloss(Xtest, ytest, model05), BrierlossA(Xtest, ytest, s, hatalpha), Brierloss(Xtest, ytest, model1))
print(Err(Xtest, ytest, model05), ErrA(Xtest, ytest, s, hatalpha), Err(Xtest, ytest, model1))

In [ ]:
def calibration_bins(s, y, alpha):
    n = s.shape[0]
    
    cnt = np.zeros(5, dtype=int)
    mean = np.zeros(5)
    obpos = np.zeros(5)

    for i in range(n):
        p1 = np.exp(-np.logaddexp(0, -alpha * s[i]))
        j = 0
        if p1 < 0.2:
            j = 0
        elif p1 < 0.4:
            j = 1
        elif p1 < 0.6:
            j = 2
        elif p1 < 0.8:
            j = 3
        else:
            j = 4
            
        cnt[j] += 1
        mean[j] += p1
        obpos[j] += y[i]

    for j in range(5):
        if cnt[j] > 0:
            mean[j] /= cnt[j]
            obpos[j] /= cnt[j]
        else:
            mean[j] = np.nan
            obpos[j] = np.nan

    return cnt, mean, obpos

# Before temperature scaling
cnt_before, mean_before, pos_before = calibration_bins(
    s, ytest, alpha=1.0
)

# After temperature scaling
cnt_after, mean_after, pos_after = calibration_bins(
    s, ytest, alpha=hatalpha
)

print(cnt_before, mean_before, pos_before)
print(cnt_after, mean_after, pos_after)